In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import shutil
import json

# Paths
SRC = "/kaggle/input/globular-clusters"   # replace with your dataset input name
DST = "/kaggle/working/train_lora"
IMAGES_OUT = os.path.join(DST, "images")

# Make output directories
os.makedirs(IMAGES_OUT, exist_ok=True)

# Prepare metadata list
metadata = []

# Iterate through images
for fn in sorted(os.listdir(SRC)):
    if fn.lower().endswith((".png", ".jpg", ".jpeg")):
        src_img = os.path.join(SRC, fn)
        dst_img = os.path.join(IMAGES_OUT, fn)
        shutil.copy(src_img, dst_img)

        # Try to read a caption file if exists
        base = os.path.splitext(fn)[0]
        cap1 = os.path.join(SRC, fn + ".txt")
        cap2 = os.path.join(SRC, base + ".txt")
        if os.path.exists(cap1):
            caption = open(cap1, "r", encoding="utf-8").read().strip()
        elif os.path.exists(cap2):
            caption = open(cap2, "r", encoding="utf-8").read().strip()
        else:
            caption = "a globular cluster in space"  # fallback caption

        # Add to metadata
        metadata.append({
            "filename": f"images/{fn}",
            "text": caption
        })

# Save metadata as JSON (LoRA expects JSON, not JSONL)
meta_path = os.path.join(DST, "metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Copied {len(metadata)} images to {IMAGES_OUT}")
print(f"Metadata saved to {meta_path}")


In [ ]:
import os
import shutil
import json

# Paths
SRC = "/kaggle/input/globular-clusters"  # your uploaded dataset
DST = "/kaggle/working/train_lora"
IMAGES_OUT = os.path.join(DST, "images")

# Make output directories
os.makedirs(IMAGES_OUT, exist_ok=True)

# Prepare metadata list
metadata = []

# Iterate through images
for fn in sorted(os.listdir(SRC)):
    if fn.lower().endswith((".png", ".jpg", ".jpeg")):
        src_img = os.path.join(SRC, fn)
        dst_img = os.path.join(IMAGES_OUT, fn)
        shutil.copy(src_img, dst_img)

        # Read caption if exists
        base = os.path.splitext(fn)[0]
        cap1 = os.path.join(SRC, fn + ".txt")
        cap2 = os.path.join(SRC, base + ".txt")
        if os.path.exists(cap1):
            caption = open(cap1, "r", encoding="utf-8").read().strip()
        elif os.path.exists(cap2):
            caption = open(cap2, "r", encoding="utf-8").read().strip()
        else:
            caption = "a globular cluster in space"

        # Add entry to metadata
        metadata.append({
            "filename": f"images/{fn}",
            "text": caption
        })

# Save metadata as JSON
meta_path = os.path.join(DST, "metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Copied {len(metadata)} images to {IMAGES_OUT}")
print(f"Metadata saved to {meta_path}")


In [ ]:
# Load metadata to confirm structure
with open(meta_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("First 3 entries in metadata:")
for entry in data[:3]:
    print(entry)


In [ ]:
!pip install --upgrade diffusers transformers accelerate safetensors

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-2-1-base" \
  --train_data_dir="/kaggle/working/train_lora" \
  --output_dir="/kaggle/working/lora_output" \
  --revision="fp16" \
  --mixed_precision="fp16" \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --max_train_steps=400 \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --save_every_n_steps=100 \
  --seed=42


In [ ]:
%%writefile /kaggle/working/train_text_to_image_lora.py
# coding=utf-8
# Copyright 2023 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""Fine-tuning script for Stable Diffusion for text2image with support for LoRA."""

import argparse
import logging
import math
import os
import random
from pathlib import Path

import datasets
import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration, set_seed
from datasets import load_dataset
from huggingface_hub import create_repo, upload_folder
from packaging import version
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

import diffusers
from diffusers import AutoencoderKL, DDPMScheduler, DiffusionPipeline, UNet2DConditionModel
from diffusers.loaders import AttnProcsLayers
from diffusers.models.attention_processor import LoRAAttnProcessor
from diffusers.optimization import get_scheduler
from diffusers.utils import check_min_version, is_wandb_available
from diffusers.utils.import_utils import is_xformers_available


# Will error if the minimal version of diffusers is not installed. Remove at your own risks.
check_min_version("0.15.0.dev0")

logger = get_logger(__name__, log_level="INFO")


def save_model_card(repo_id: str, images=None, base_model=str, dataset_name=str, repo_folder=None):
    img_str = ""
    for i, image in enumerate(images):
        image.save(os.path.join(repo_folder, f"image_{i}.png"))
        img_str += f"![img_{i}](./image_{i}.png)\n"

    yaml = f"""
---
license: creativeml-openrail-m
base_model: {base_model}
tags:
- stable-diffusion
- stable-diffusion-diffusers
- text-to-image
- diffusers
- lora
inference: true
---
    """
    model_card = f"""
# LoRA text2image fine-tuning - {repo_id}
These are LoRA adaption weights for {base_model}. The weights were fine-tuned on the {dataset_name} dataset. You can find some example images in the following. \n
{img_str}
"""
    with open(os.path.join(repo_folder, "README.md"), "w") as f:
        f.write(yaml + model_card)


def parse_args():
    parser = argparse.ArgumentParser(description="Simple example of a training script.")
    parser.add_argument(
        "--pretrained_model_name_or_path",
        type=str,
        default=None,
        required=True,
        help="Path to pretrained model or model identifier from huggingface.co/models.",
    )
    parser.add_argument(
        "--revision",
        type=str,
        default=None,
        required=False,
        help="Revision of pretrained model identifier from huggingface.co/models.",
    )
    parser.add_argument(
        "--dataset_name",
        type=str,
        default=None,
        help=(
            "The name of the Dataset (from the HuggingFace hub) to train on (could be your own, possibly private,"
            " dataset). It can also be a path pointing to a local copy of a dataset in your filesystem,"
            " or to a folder containing files that 🤗 Datasets can understand."
        ),
    )
    parser.add_argument(
        "--dataset_config_name",
        type=str,
        default=None,
        help="The config of the Dataset, leave as None if there's only one config.",
    )
    parser.add_argument(
        "--train_data_dir",
        type=str,
        default=None,
        help=(
            "A folder containing the training data. Folder contents must follow the structure described in"
            " https://huggingface.co/docs/datasets/image_dataset#imagefolder. In particular, a `metadata.jsonl` file"
            " must exist to provide the captions for the images. Ignored if `dataset_name` is specified."
        ),
    )
    parser.add_argument(
        "--image_column", type=str, default="image", help="The column of the dataset containing an image."
    )
    parser.add_argument(
        "--caption_column",
        type=str,
        default="text",
        help="The column of the dataset containing a caption or a list of captions.",
    )
    parser.add_argument(
        "--validation_prompt", type=str, default=None, help="A prompt that is sampled during training for inference."
    )
    parser.add_argument(
        "--num_validation_images",
        type=int,
        default=4,
        help="Number of images that should be generated during validation with `validation_prompt`.",
    )
    parser.add_argument(
        "--validation_epochs",
        type=int,
        default=1,
        help=(
            "Run fine-tuning validation every X epochs. The validation process consists of running the prompt"
            " `args.validation_prompt` multiple times: `args.num_validation_images`."
        ),
    )
    parser.add_argument(
        "--max_train_samples",
        type=int,
        default=None,
        help=(
            "For debugging purposes or quicker training, truncate the number of training examples to this "
            "value if set."
        ),
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default="sd-model-finetuned-lora",
        help="The output directory where the model predictions and checkpoints will be written.",
    )
    parser.add_argument(
        "--cache_dir",
        type=str,
        default=None,
        help="The directory where the downloaded models and datasets will be stored.",
    )
    parser.add_argument("--seed", type=int, default=None, help="A seed for reproducible training.")
    parser.add_argument(
        "--resolution",
        type=int,
        default=512,
        help=(
            "The resolution for input images, all the images in the train/validation dataset will be resized to this"
            " resolution"
        ),
    )
    parser.add_argument(
        "--center_crop",
        default=False,
        action="store_true",
        help=(
            "Whether to center crop the input images to the resolution. If not set, the images will be randomly"
            " cropped. The images will be resized to the resolution first before cropping."
        ),
    )
    parser.add_argument(
        "--random_flip",
        action="store_true",
        help="whether to randomly flip images horizontally",
    )
    parser.add_argument(
        "--train_batch_size", type=int, default=16, help="Batch size (per device) for the training dataloader."
    )
    parser.add_argument("--num_train_epochs", type=int, default=100)
    parser.add_argument(
        "--max_train_steps",
        type=int,
        default=None,
        help="Total number of training steps to perform.  If provided, overrides num_train_epochs.",
    )
    parser.add_argument(
        "--gradient_accumulation_steps",
        type=int,
        default=1,
        help="Number of updates steps to accumulate before performing a backward/update pass.",
    )
    parser.add_argument(
        "--gradient_checkpointing",
        action="store_true",
        help="Whether or not to use gradient checkpointing to save memory at the expense of slower backward pass.",
    )
    parser.add_argument(
        "--learning_rate",
        type=float,
        default=1e-4,
        help="Initial learning rate (after the potential warmup period) to use.",
    )
    parser.add_argument(
        "--scale_lr",
        action="store_true",
        default=False,
        help="Scale the learning rate by the number of GPUs, gradient accumulation steps, and batch size.",
    )
    parser.add_argument(
        "--lr_scheduler",
        type=str,
        default="constant",
        help=(
            'The scheduler type to use. Choose between ["linear", "cosine", "cosine_with_restarts", "polynomial",'
            ' "constant", "constant_with_warmup"]'
        ),
    )
    parser.add_argument(
        "--lr_warmup_steps", type=int, default=500, help="Number of steps for the warmup in the lr scheduler."
    )
    parser.add_argument(
        "--use_8bit_adam", action="store_true", help="Whether or not to use 8-bit Adam from bitsandbytes."
    )
    parser.add_argument(
        "--allow_tf32",
        action="store_true",
        help=(
            "Whether or not to allow TF32 on Ampere GPUs. Can be used to speed up training. For more information, see"
            " https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices"
        ),
    )
    parser.add_argument(
        "--dataloader_num_workers",
        type=int,
        default=0,
        help=(
            "Number of subprocesses to use for data loading. 0 means that the data will be loaded in the main process."
        ),
    )
    parser.add_argument("--adam_beta1", type=float, default=0.9, help="The beta1 parameter for the Adam optimizer.")
    parser.add_argument("--adam_beta2", type=float, default=0.999, help="The beta2 parameter for the Adam optimizer.")
    parser.add_argument("--adam_weight_decay", type=float, default=1e-2, help="Weight decay to use.")
    parser.add_argument("--adam_epsilon", type=float, default=1e-08, help="Epsilon value for the Adam optimizer")
    parser.add_argument("--max_grad_norm", default=1.0, type=float, help="Max gradient norm.")
    parser.add_argument("--push_to_hub", action="store_true", help="Whether or not to push the model to the Hub.")
    parser.add_argument("--hub_token", type=str, default=None, help="The token to use to push to the Model Hub.")
    parser.add_argument(
        "--hub_model_id",
        type=str,
        default=None,
        help="The name of the repository to keep in sync with the local `output_dir`.",
    )
    parser.add_argument(
        "--logging_dir",
        type=str,
        default="logs",
        help=(
            "[TensorBoard](https://www.tensorflow.org/tensorboard) log directory. Will default to"
            " *output_dir/runs/**CURRENT_DATETIME_HOSTNAME***."
        ),
    )
    parser.add_argument(
        "--mixed_precision",
        type=str,
        default=None,
        choices=["no", "fp16", "bf16"],
        help=(
            "Whether to use mixed precision. Choose between fp16 and bf16 (bfloat16). Bf16 requires PyTorch >="
            " 1.10.and an Nvidia Ampere GPU.  Default to the value of accelerate config of the current system or the"
            " flag passed with the `accelerate.launch` command. Use this argument to override the accelerate config."
        ),
    )
    parser.add_argument(
        "--report_to",
        type=str,
        default="tensorboard",
        help=(
            'The integration to report the results and logs to. Supported platforms are `"tensorboard"`'
            ' (default), `"wandb"` and `"comet_ml"`. Use `"all"` to report to all integrations.'
        ),
    )
    parser.add_argument("--local_rank", type=int, default=-1, help="For distributed training: local_rank")
    parser.add_argument(
        "--checkpointing_steps",
        type=int,
        default=500,
        help=(
            "Save a checkpoint of the training state every X updates. These checkpoints are only suitable for resuming"
            " training using `--resume_from_checkpoint`."
        ),
    )
    parser.add_argument(
        "--checkpoints_total_limit",
        type=int,
        default=None,
        help=(
            "Max number of checkpoints to store. Passed as `total_limit` to the `Accelerator` `ProjectConfiguration`."
            " See Accelerator::save_state https://huggingface.co/docs/accelerate/package_reference/accelerator#accelerate.Accelerator.save_state"
            " for more docs"
        ),
    )
    parser.add_argument(
        "--resume_from_checkpoint",
        type=str,
        default=None,
        help=(
            "Whether training should be resumed from a previous checkpoint. Use a path saved by"
            ' `--checkpointing_steps`, or `"latest"` to automatically select the last available checkpoint.'
        ),
    )
    parser.add_argument(
        "--enable_xformers_memory_efficient_attention", action="store_true", help="Whether or not to use xformers."
    )
    parser.add_argument("--noise_offset", type=float, default=0, help="The scale of noise offset.")

    args = parser.parse_args()
    env_local_rank = int(os.environ.get("LOCAL_RANK", -1))
    if env_local_rank != -1 and env_local_rank != args.local_rank:
        args.local_rank = env_local_rank

    # Sanity checks
    if args.dataset_name is None and args.train_data_dir is None:
        raise ValueError("Need either a dataset name or a training folder.")

    return args


DATASET_NAME_MAPPING = {
    "lambdalabs/pokemon-blip-captions": ("image", "text"),
}


def main():
    args = parse_args()
    logging_dir = os.path.join(args.output_dir, args.logging_dir)

    accelerator_project_config = ProjectConfiguration(total_limit=args.checkpoints_total_limit)

    accelerator = Accelerator(
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    mixed_precision=args.mixed_precision,
    log_with=[args.report_to] if args.report_to else [],
    logging_dir=args.logging_dir if args.logging_dir else None,
        # Remove project_config if not defined
    )
    if args.report_to == "wandb":
        if not is_wandb_available():
            raise ImportError("Make sure to install wandb if you want to use it for logging during training.")
        import wandb

    # Make one log on every process with the configuration for debugging.
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
    )
    logger.info(accelerator.state, main_process_only=False)
    if accelerator.is_local_main_process:
        datasets.utils.logging.set_verbosity_warning()
        transformers.utils.logging.set_verbosity_warning()
        diffusers.utils.logging.set_verbosity_info()
    else:
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()
        diffusers.utils.logging.set_verbosity_error()

    # If passed along, set the training seed now.
    if args.seed is not None:
        set_seed(args.seed)

    # Handle the repository creation
    if accelerator.is_main_process:
        if args.output_dir is not None:
            os.makedirs(args.output_dir, exist_ok=True)

        if args.push_to_hub:
            repo_id = create_repo(
                repo_id=args.hub_model_id or Path(args.output_dir).name, exist_ok=True, token=args.hub_token
            ).repo_id
    # Load scheduler, tokenizer and models.
    noise_scheduler = DDPMScheduler.from_pretrained(args.pretrained_model_name_or_path, subfolder="scheduler")
    tokenizer = CLIPTokenizer.from_pretrained(
        args.pretrained_model_name_or_path, subfolder="tokenizer", revision=args.revision
    )
    text_encoder = CLIPTextModel.from_pretrained(
        args.pretrained_model_name_or_path, subfolder="text_encoder", revision=args.revision
    )
    vae = AutoencoderKL.from_pretrained(args.pretrained_model_name_or_path, subfolder="vae", revision=args.revision)
    unet = UNet2DConditionModel.from_pretrained(
        args.pretrained_model_name_or_path, subfolder="unet", revision=args.revision
    )
    # freeze parameters of models to save more memory
    unet.requires_grad_(False)
    vae.requires_grad_(False)

    text_encoder.requires_grad_(False)

    # For mixed precision training we cast the text_encoder and vae weights to half-precision
    # as these models are only used for inference, keeping weights in full precision is not required.
    weight_dtype = torch.float32
    if accelerator.mixed_precision == "fp16":
        weight_dtype = torch.float16
    elif accelerator.mixed_precision == "bf16":
        weight_dtype = torch.bfloat16

    # Move unet, vae and text_encoder to device and cast to weight_dtype
    unet.to(accelerator.device, dtype=weight_dtype)
    vae.to(accelerator.device, dtype=weight_dtype)
    text_encoder.to(accelerator.device, dtype=weight_dtype)

    # now we will add new LoRA weights to the attention layers
    # It's important to realize here how many attention weights will be added and of which sizes
    # The sizes of the attention layers consist only of two different variables:
    # 1) - the "hidden_size", which is increased according to `unet.config.block_out_channels`.
    # 2) - the "cross attention size", which is set to `unet.config.cross_attention_dim`.

    # Let's first see how many attention processors we will have to set.
    # For Stable Diffusion, it should be equal to:
    # - down blocks (2x attention layers) * (2x transformer layers) * (3x down blocks) = 12
    # - mid blocks (2x attention layers) * (1x transformer layers) * (1x mid blocks) = 2
    # - up blocks (2x attention layers) * (3x transformer layers) * (3x down blocks) = 18
    # => 32 layers

    # Set correct lora layers
    lora_attn_procs = {}
    for name in unet.attn_processors.keys():
        cross_attention_dim = None if name.endswith("attn1.processor") else unet.config.cross_attention_dim
        if name.startswith("mid_block"):
            hidden_size = unet.config.block_out_channels[-1]
        elif name.startswith("up_blocks"):
            block_id = int(name[len("up_blocks.")])
            hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
        elif name.startswith("down_blocks"):
            block_id = int(name[len("down_blocks.")])
            hidden_size = unet.config.block_out_channels[block_id]

        lora_attn_procs[name] = LoRAAttnProcessor(hidden_size=hidden_size, cross_attention_dim=cross_attention_dim)

    unet.set_attn_processor(lora_attn_procs)

    if args.enable_xformers_memory_efficient_attention:
        if is_xformers_available():
            import xformers

            xformers_version = version.parse(xformers.__version__)
            if xformers_version == version.parse("0.0.16"):
                logger.warn(
                    "xFormers 0.0.16 cannot be used for training in some GPUs. If you observe problems during training, please update xFormers to at least 0.0.17. See https://huggingface.co/docs/diffusers/main/en/optimization/xformers for more details."
                )
            unet.enable_xformers_memory_efficient_attention()
        else:
            raise ValueError("xformers is not available. Make sure it is installed correctly")

    lora_layers = AttnProcsLayers(unet.attn_processors)

    # Enable TF32 for faster training on Ampere GPUs,
    # cf https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
    if args.allow_tf32:
        torch.backends.cuda.matmul.allow_tf32 = True

    if args.scale_lr:
        args.learning_rate = (
            args.learning_rate * args.gradient_accumulation_steps * args.train_batch_size * accelerator.num_processes
        )

    # Initialize the optimizer
    if args.use_8bit_adam:
        try:
            import bitsandbytes as bnb
        except ImportError:
            raise ImportError(
                "Please install bitsandbytes to use 8-bit Adam. You can do so by running `pip install bitsandbytes`"
            )

        optimizer_cls = bnb.optim.AdamW8bit
    else:
        optimizer_cls = torch.optim.AdamW

    optimizer = optimizer_cls(
        lora_layers.parameters(),
        lr=args.learning_rate,
        betas=(args.adam_beta1, args.adam_beta2),
        weight_decay=args.adam_weight_decay,
        eps=args.adam_epsilon,
    )

    # Get the datasets: you can either provide your own training and evaluation files (see below)
    # or specify a Dataset from the hub (the dataset will be downloaded automatically from the datasets Hub).

    # In distributed training, the load_dataset function guarantees that only one local process can concurrently
    # download the dataset.
    if args.dataset_name is not None:
        # Downloading and loading a dataset from the hub.
        dataset = load_dataset(
            args.dataset_name,
            args.dataset_config_name,
            cache_dir=args.cache_dir,
        )
    else:
        data_files = {}
        if args.train_data_dir is not None:
            data_files["train"] = os.path.join(args.train_data_dir, "**")
        dataset = load_dataset(
            "imagefolder",
            data_files=data_files,
            cache_dir=args.cache_dir,
        )
        # See more about loading custom images at
        # https://huggingface.co/docs/datasets/v2.4.0/en/image_load#imagefolder

    # Preprocessing the datasets.
    # We need to tokenize inputs and targets.
    column_names = dataset["train"].column_names

    # 6. Get the column names for input/target.
    dataset_columns = DATASET_NAME_MAPPING.get(args.dataset_name, None)
    if args.image_column is None:
        image_column = dataset_columns[0] if dataset_columns is not None else column_names[0]
    else:
        image_column = args.image_column
        if image_column not in column_names:
            raise ValueError(
                f"--image_column' value '{args.image_column}' needs to be one of: {', '.join(column_names)}"
            )
    if args.caption_column is None:
        caption_column = dataset_columns[1] if dataset_columns is not None else column_names[1]
    else:
        caption_column = args.caption_column
        if caption_column not in column_names:
            raise ValueError(
                f"--caption_column' value '{args.caption_column}' needs to be one of: {', '.join(column_names)}"
            )

    # Preprocessing the datasets.
    # We need to tokenize input captions and transform the images.
    def tokenize_captions(examples, is_train=True):
        captions = []
        for caption in examples[caption_column]:
            if isinstance(caption, str):
                captions.append(caption)
            elif isinstance(caption, (list, np.ndarray)):
                # take a random caption if there are multiple
                captions.append(random.choice(caption) if is_train else caption[0])
            else:
                raise ValueError(
                    f"Caption column `{caption_column}` should contain either strings or lists of strings."
                )
        inputs = tokenizer(
            captions, max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt"
        )
        return inputs.input_ids

    # Preprocessing the datasets.
    train_transforms = transforms.Compose(
        [
            transforms.Resize(args.resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(args.resolution) if args.center_crop else transforms.RandomCrop(args.resolution),
            transforms.RandomHorizontalFlip() if args.random_flip else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ]
    )

    def preprocess_train(examples):
        images = [image.convert("RGB") for image in examples[image_column]]
        examples["pixel_values"] = [train_transforms(image) for image in images]
        examples["input_ids"] = tokenize_captions(examples)
        return examples

    with accelerator.main_process_first():
        if args.max_train_samples is not None:
            dataset["train"] = dataset["train"].shuffle(seed=args.seed).select(range(args.max_train_samples))
        # Set the training transforms
        train_dataset = dataset["train"].with_transform(preprocess_train)

    def collate_fn(examples):
        pixel_values = torch.stack([example["pixel_values"] for example in examples])
        pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()
        input_ids = torch.stack([example["input_ids"] for example in examples])
        return {"pixel_values": pixel_values, "input_ids": input_ids}

    # DataLoaders creation:
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        shuffle=True,
        collate_fn=collate_fn,
        batch_size=args.train_batch_size,
        num_workers=args.dataloader_num_workers,
    )

    # Scheduler and math around the number of training steps.
    overrode_max_train_steps = False
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
    if args.max_train_steps is None:
        args.max_train_steps = args.num_train_epochs * num_update_steps_per_epoch
        overrode_max_train_steps = True

    lr_scheduler = get_scheduler(
        args.lr_scheduler,
        optimizer=optimizer,
        num_warmup_steps=args.lr_warmup_steps * args.gradient_accumulation_steps,
        num_training_steps=args.max_train_steps * args.gradient_accumulation_steps,
    )

    # Prepare everything with our `accelerator`.
    lora_layers, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        lora_layers, optimizer, train_dataloader, lr_scheduler
    )

    # We need to recalculate our total training steps as the size of the training dataloader may have changed.
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
    if overrode_max_train_steps:
        args.max_train_steps = args.num_train_epochs * num_update_steps_per_epoch
    # Afterwards we recalculate our number of training epochs
    args.num_train_epochs = math.ceil(args.max_train_steps / num_update_steps_per_epoch)

    # We need to initialize the trackers we use, and also store our configuration.
    # The trackers initializes automatically on the main process.
    if accelerator.is_main_process:
        accelerator.init_trackers("text2image-fine-tune", config=vars(args))

    # Train!
    total_batch_size = args.train_batch_size * accelerator.num_processes * args.gradient_accumulation_steps

    logger.info("***** Running training *****")
    logger.info(f"  Num examples = {len(train_dataset)}")
    logger.info(f"  Num Epochs = {args.num_train_epochs}")
    logger.info(f"  Instantaneous batch size per device = {args.train_batch_size}")
    logger.info(f"  Total train batch size (w. parallel, distributed & accumulation) = {total_batch_size}")
    logger.info(f"  Gradient Accumulation steps = {args.gradient_accumulation_steps}")
    logger.info(f"  Total optimization steps = {args.max_train_steps}")
    global_step = 0
    first_epoch = 0

    # Potentially load in the weights and states from a previous save
    if args.resume_from_checkpoint:
        if args.resume_from_checkpoint != "latest":
            path = os.path.basename(args.resume_from_checkpoint)
        else:
            # Get the most recent checkpoint
            dirs = os.listdir(args.output_dir)
            dirs = [d for d in dirs if d.startswith("checkpoint")]
            dirs = sorted(dirs, key=lambda x: int(x.split("-")[1]))
            path = dirs[-1] if len(dirs) > 0 else None

        if path is None:
            accelerator.print(
                f"Checkpoint '{args.resume_from_checkpoint}' does not exist. Starting a new training run."
            )
            args.resume_from_checkpoint = None
        else:
            accelerator.print(f"Resuming from checkpoint {path}")
            accelerator.load_state(os.path.join(args.output_dir, path))
            global_step = int(path.split("-")[1])

            resume_global_step = global_step * args.gradient_accumulation_steps
            first_epoch = global_step // num_update_steps_per_epoch
            resume_step = resume_global_step % (num_update_steps_per_epoch * args.gradient_accumulation_steps)

    # Only show the progress bar once on each machine.
    progress_bar = tqdm(range(global_step, args.max_train_steps), disable=not accelerator.is_local_main_process)
    progress_bar.set_description("Steps")

    for epoch in range(first_epoch, args.num_train_epochs):
        unet.train()
        train_loss = 0.0
        for step, batch in enumerate(train_dataloader):
            # Skip steps until we reach the resumed step
            if args.resume_from_checkpoint and epoch == first_epoch and step < resume_step:
                if step % args.gradient_accumulation_steps == 0:
                    progress_bar.update(1)
                continue

            with accelerator.accumulate(unet):
                # Convert images to latent space
                latents = vae.encode(batch["pixel_values"].to(dtype=weight_dtype)).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

                # Sample noise that we'll add to the latents
                noise = torch.randn_like(latents)
                if args.noise_offset:
                    # https://www.crosslabs.org//blog/diffusion-with-offset-noise
                    noise += args.noise_offset * torch.randn(
                        (latents.shape[0], latents.shape[1], 1, 1), device=latents.device
                    )

                bsz = latents.shape[0]
                # Sample a random timestep for each image
                timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (bsz,), device=latents.device)
                timesteps = timesteps.long()

                # Add noise to the latents according to the noise magnitude at each timestep
                # (this is the forward diffusion process)
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # Get the text embedding for conditioning
                encoder_hidden_states = text_encoder(batch["input_ids"])[0]

                # Get the target for loss depending on the prediction type
                if noise_scheduler.config.prediction_type == "epsilon":
                    target = noise
                elif noise_scheduler.config.prediction_type == "v_prediction":
                    target = noise_scheduler.get_velocity(latents, noise, timesteps)
                else:
                    raise ValueError(f"Unknown prediction type {noise_scheduler.config.prediction_type}")

                # Predict the noise residual and compute loss
                model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
                loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")

                # Gather the losses across all processes for logging (if we use distributed training).
                avg_loss = accelerator.gather(loss.repeat(args.train_batch_size)).mean()
                train_loss += avg_loss.item() / args.gradient_accumulation_steps

                # Backpropagate
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    params_to_clip = lora_layers.parameters()
                    accelerator.clip_grad_norm_(params_to_clip, args.max_grad_norm)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            # Checks if the accelerator has performed an optimization step behind the scenes
            if accelerator.sync_gradients:
                progress_bar.update(1)
                global_step += 1
                accelerator.log({"train_loss": train_loss}, step=global_step)
                train_loss = 0.0

                if global_step % args.checkpointing_steps == 0:
                    if accelerator.is_main_process:
                        save_path = os.path.join(args.output_dir, f"checkpoint-{global_step}")
                        accelerator.save_state(save_path)
                        logger.info(f"Saved state to {save_path}")

            logs = {"step_loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0]}
            progress_bar.set_postfix(**logs)

            if global_step >= args.max_train_steps:
                break

        if accelerator.is_main_process:
            if args.validation_prompt is not None and epoch % args.validation_epochs == 0:
                logger.info(
                    f"Running validation... \n Generating {args.num_validation_images} images with prompt:"
                    f" {args.validation_prompt}."
                )
                # create pipeline
                pipeline = DiffusionPipeline.from_pretrained(
                    args.pretrained_model_name_or_path,
                    unet=accelerator.unwrap_model(unet),
                    revision=args.revision,
                    torch_dtype=weight_dtype,
                )
                pipeline = pipeline.to(accelerator.device)
                pipeline.set_progress_bar_config(disable=True)

                # run inference
                generator = torch.Generator(device=accelerator.device).manual_seed(args.seed)
                images = []
                for _ in range(args.num_validation_images):
                    images.append(
                        pipeline(args.validation_prompt, num_inference_steps=30, generator=generator).images[0]
                    )

                for tracker in accelerator.trackers:
                    if tracker.name == "tensorboard":
                        np_images = np.stack([np.asarray(img) for img in images])
                        tracker.writer.add_images("validation", np_images, epoch, dataformats="NHWC")
                    if tracker.name == "wandb":
                        tracker.log(
                            {
                                "validation": [
                                    wandb.Image(image, caption=f"{i}: {args.validation_prompt}")
                                    for i, image in enumerate(images)
                                ]
                            }
                        )

                del pipeline
                torch.cuda.empty_cache()

    # Save the lora layers
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        unet = unet.to(torch.float32)
        unet.save_attn_procs(args.output_dir)

        if args.push_to_hub:
            save_model_card(
                repo_id,
                images=images,
                base_model=args.pretrained_model_name_or_path,
                dataset_name=args.dataset_name,
                repo_folder=args.output_dir,
            )
            upload_folder(
                repo_id=repo_id,
                folder_path=args.output_dir,
                commit_message="End of training",
                ignore_patterns=["step_*", "epoch_*"],
            )

    # Final inference
    # Load previous pipeline
    pipeline = DiffusionPipeline.from_pretrained(
        args.pretrained_model_name_or_path, revision=args.revision, torch_dtype=weight_dtype
    )
    pipeline = pipeline.to(accelerator.device)

    # load attention processors
    pipeline.unet.load_attn_procs(args.output_dir)

    # run inference
    generator = torch.Generator(device=accelerator.device).manual_seed(args.seed)
    images = []
    for _ in range(args.num_validation_images):
        images.append(pipeline(args.validation_prompt, num_inference_steps=30, generator=generator).images[0])

    if accelerator.is_main_process:
        for tracker in accelerator.trackers:
            if tracker.name == "tensorboard":
                np_images = np.stack([np.asarray(img) for img in images])
                tracker.writer.add_images("test", np_images, epoch, dataformats="NHWC")
            if tracker.name == "wandb":
                tracker.log(
                    {
                        "test": [
                            wandb.Image(image, caption=f"{i}: {args.validation_prompt}")
                            for i, image in enumerate(images)
                        ]
                    }
                )

    accelerator.end_training()


if __name__ == "__main__":
    main()


In [ ]:
import json
from pathlib import Path

metadata_path = Path("../input/your-dataset/metadata.json")

# Load the original metadata
with open(metadata_path, "r") as f:
    data = json.load(f)

# Update filenames to point to the input folder
for item in data:
    item["filename"] = f"../input/your-dataset/images/{Path(item['filename']).name}"

# Save it to working directory with the same name
output_path = Path("/kaggle/working/metadata.json")
with open(output_path, "w") as f:
    json.dump(data, f, indent=2)


import json
from pathlib import Path

# Paths
metadata_file = Path("/kaggle/working/train_lora/metadata.json")
images_folder = Path("/kaggle/working/train_lora/images")

# Load metadata
with open(metadata_file, "r") as f:
    data = json.load(f)

# Update metadata
for item in data:
    # Ensure the key is 'filename'
    if "image" in item:
        item["filename"] = str(images_folder / Path(item["image"]).name)
        del item["image"]
    elif "filename" in item:
        item["filename"] = str(images_folder / Path(item["filename"]).name)

# Save fixed metadata back
fixed_metadata_file = Path("/kaggle/working/train_lora/metadata_fixed.json")
with open(fixed_metadata_file, "w") as f:
    json.dump(data, f, indent=2)

print(f"Updated metadata saved to {fixed_metadata_file}")


!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --max_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# Load the base model
model_id = "runwayml/stable-diffusion-v1-5"

# Load pipeline with LoRA weights
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# Load the LoRA attention processors
pipe.unet.load_attn_procs("/kaggle/working/sd_lora_output")

# Move to GPU
pipe = pipe.to("cuda")

# Set your prompt
prompt = "globular star cluster, deep space, Hubble telescope"

# Generate image
image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]

# Save image
image.save("/kaggle/working/generated_image.png")

# Display in notebook
image.show()


In [ ]:
pip install peft

In [ ]:
!pip install --upgrade --no-cache-dir peft

In [ ]:
import importlib
import peft

importlib.reload(peft)

print(peft.__version__)  # confirm it's upgraded


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="image" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --num_train_epochs=10 \
    --learning_rate=1e-4 \
    --gradient_accumulation_steps=1 \
    --output_dir="/kaggle/working/output_lora" \
    --mixed_precision="fp16" \
    --logging_dir="/kaggle/working/logs_lora"


In [ ]:
!pip install --upgrade --no-cache-dir accelerate

In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --max_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip

In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --max_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip

In [ ]:
# Path to your script
script_path = "/kaggle/working/train_text_to_image_lora.py"

# Read the script
with open(script_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Modify lines in-place
new_lines = []
for line in lines:
    # Remove logging_dir from Accelerator() call
    if "Accelerator(" in line and "logging_dir" in line:
        # Remove logging_dir argument
        # This will handle a line like: Accelerator(logging_dir=args.logging_dir, ...)
        line = line.replace("logging_dir=args.logging_dir,", "")
        line = line.replace("logging_dir=args.logging_dir", "")  # in case no trailing comma
    new_lines.append(line)

# Write back the modified script
with open(script_path, "w", encoding="utf-8") as f:
    f.writelines(new_lines)

print("Script updated: removed logging_dir from Accelerator()")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --max_train_steps=400 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip

In [ ]:
# Patch the script in-place
import fileinput

script_path = "/kaggle/working/train_text_to_image_lora.py"

for line in fileinput.input(script_path, inplace=True):
    # Remove logging_dir argument from Accelerator initialization
    print(line.replace("logging_dir=logging_dir,", ""), end="")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip \
  --logging_dir="/kaggle/working/logs_lora"

In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip \
  --report_to="none"


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip


In [ ]:
# Patch train_text_to_image_lora.py for correct Accelerator usage
script_path = "/kaggle/working/train_text_to_image_lora.py"

# Read the script
with open(script_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Modify the line where Accelerator is instantiated
new_lines = []
for line in lines:
    if "Accelerator(" in line:
        # Replace any logging_dir or log_with argument with safe defaults
        new_line = '    accelerator = Accelerator(mixed_precision="fp16")\n'
        new_lines.append(new_line)
    else:
        new_lines.append(line)

# Write back the modified script
with open(script_path, "w", encoding="utf-8") as f:
    f.writelines(new_lines)

print("Script patched: Accelerator will now initialize without logging issues.")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip


In [ ]:
script_path = "/kaggle/working/train_text_to_image_lora.py"

# Read the original script
with open(script_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Edit the Accelerator initialization to remove `logging_dir` and fix indentation
for i, line in enumerate(lines):
    if "Accelerator(" in line:
        # Replace the whole block starting from this line until the closing parenthesis
        # Assumes standard indentation
        j = i
        while ")" not in lines[j]:
            j += 1
        # New block
        lines[i:j+1] = [
            "    accelerator = Accelerator(\n",
            "        mixed_precision='fp16',\n",
            "    )\n"
        ]
        break

# Save the modified script
with open(script_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print("Script fixed in-place.")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_lora" \
  --image_column="filename" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=4 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 \
  --num_train_epochs=10 \
  --output_dir="/kaggle/working/sd_lora_output" \
  --mixed_precision="fp16" \
  --center_crop \
  --random_flip


In [ ]:
# Path to your script
script_path = "/kaggle/working/train_text_to_image_lora.py"

# Read the original script
with open(script_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    # Remove logging_dir argument from Accelerator instantiation
    if "Accelerator(" in line:
        in_accelerator = True
        new_lines.append(line)
        continue
    if "gradient_accumulation_steps=" in line:
        # Ensure correct indentation and keep the line
        new_lines.append("    gradient_accumulation_steps=1,\n")
        continue
    if "logging_dir=" in line:
        # Skip any logging_dir line
        continue
    new_lines.append(line)

# Write back the fixed script
with open(script_path, "w") as f:
    f.writelines(new_lines)

print("Script patched: indentation fixed, logging_dir removed, gradient_accumulation_steps set to 1")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="filename" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --num_train_epochs=10 \
    --output_dir="/kaggle/working/sd_lora_output" \
    --mixed_precision="fp16" \
    --center_crop \
    --random_flip


In [ ]:
# Patch the script with the correct Accelerator block
import re

input_path = "/kaggle/working/train_text_to_image_lora.py"
output_path = "/kaggle/working/train_text_to_image_lora_fixed.py"

with open(input_path, "r") as f:
    code = f.read()

# Replace the Accelerator init block
code_fixed = re.sub(
    r"accelerator\s*=\s*Accelerator\([^\)]*\)",
    """accelerator = Accelerator(
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    mixed_precision=args.mixed_precision,
    log_with=args.report_to if args.report_to else [],
    logging_dir=args.logging_dir if args.logging_dir else None
)""",
    code,
    flags=re.MULTILINE
)

with open(output_path, "w") as f:
    f.write(code_fixed)

print("Patched script saved to train_text_to_image_lora_fixed.py")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora_fixed.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="filename" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --num_train_epochs=10 \
    --output_dir="/kaggle/working/sd_lora_output" \
    --mixed_precision="fp16" \
    --center_crop \
    --random_flip \
    --logging_dir="/kaggle/working/logs_lora" \
    --report_to="tensorboard"


In [ ]:
# Fully overwrite the Accelerator block safely
fixed_script_path = "/kaggle/working/train_text_to_image_lora_fixed.py"

with open("/kaggle/working/train_text_to_image_lora.py", "r") as f:
    lines = f.readlines()

new_lines = []
skip_block = False
for line in lines:
    if "accelerator = Accelerator(" in line:
        # Skip original block
        skip_block = True
        # Insert fixed block
        new_lines.append("    accelerator = Accelerator(\n")
        new_lines.append("        gradient_accumulation_steps=args.gradient_accumulation_steps,\n")
        new_lines.append("        mixed_precision=args.mixed_precision,\n")
        new_lines.append("        log_with=[args.report_to] if args.report_to else [],\n")
        new_lines.append("        logging_dir=args.logging_dir if args.logging_dir else None\n")
        new_lines.append("    )\n")
        continue
    if skip_block:
        if ")" in line:  # end of original Accelerator call
            skip_block = False
        continue
    new_lines.append(line)

with open(fixed_script_path, "w") as f:
    f.writelines(new_lines)

print(f"Fixed script saved at: {fixed_script_path}")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora_fixed.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="filename" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --num_train_epochs=10 \
    --output_dir="/kaggle/working/sd_lora_output" \
    --mixed_precision="fp16" \
    --center_crop \
    --random_flip \
    --logging_dir="/kaggle/working/logs_lora" \
    --report_to="tensorboard"

In [ ]:
# Path to original and fixed script
original_path = "/kaggle/working/train_text_to_image_lora.py"
fixed_path = "/kaggle/working/train_text_to_image_lora_fixed.py"

# Read original script
with open(original_path, "r") as f:
    lines = f.readlines()

new_lines = []
skip = False

for line in lines:
    if "accelerator = Accelerator(" in line:
        skip = True  # skip original block
        # Insert fully formatted Accelerator block (4 spaces per indent)
        new_lines.append("    accelerator = Accelerator(\n")
        new_lines.append("        gradient_accumulation_steps=args.gradient_accumulation_steps,\n")
        new_lines.append("        mixed_precision=args.mixed_precision,\n")
        new_lines.append("        log_with=[args.report_to] if args.report_to else [],\n")
        new_lines.append("        logging_dir=args.logging_dir if args.logging_dir else None\n")
        new_lines.append("    )\n")
        continue
    if skip:
        # Stop skipping when original block ends
        if line.strip().endswith(")"):
            skip = False
        continue
    new_lines.append(line)

# Save fixed script
with open(fixed_path, "w") as f:
    f.writelines(new_lines)

print(f"Fixed script saved at: {fixed_path}")


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora_fixed.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="filename" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --num_train_epochs=10 \
    --output_dir="/kaggle/working/sd_lora_output" \
    --mixed_precision="fp16" \
    --center_crop \
    --random_flip \
    --logging_dir="/kaggle/working/logs_lora" \
    --report_to="tensorboard"


In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora_fixed.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="filename" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --num_train_epochs=10 \
    --output_dir="/kaggle/working/sd_lora_output" \
    --mixed_precision="fp16" \
    --center_crop \
    --random_flip \
    --logging_dir="/kaggle/working/logs_lora" \
    --report_to="tensorboard"


In [ ]:
def main():
    # Initialize accelerator
    accelerator = Accelerator(
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        mixed_precision=args.mixed_precision,
        log_with=[args.report_to] if args.report_to else [],
        logging_dir=args.logging_dir if args.logging_dir else None,
    )

    # Your training setup continues below...
    # (data loading, model setup, optimizer, etc.)

In [ ]:
!accelerate launch /kaggle/working/train_text_to_image_lora_fixed.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="/kaggle/working/train_lora" \
    --image_column="filename" \
    --caption_column="text" \
    --resolution=512 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=1 \
    --learning_rate=1e-4 \
    --num_train_epochs=10 \
    --output_dir="/kaggle/working/sd_lora_output" \
    --mixed_precision="fp16" \
    --center_crop \
    --random_flip \
    --logging_dir="/kaggle/working/logs_lora" \
    --report_to="tensorboard"


In [ ]:
!pip install --upgrade diffusers transformers accelerate safetensors bitsandbytes

In [ ]:
from diffusers import DiffusionPipeline
from accelerate import Accelerator
import torch

from diffusers import StableDiffusionPipeline
from diffusers.training_utils import enable_full_determinism
from diffusers import StableDiffusionPipeline, DDPMScheduler
from diffusers import StableDiffusionTrainer

# Accelerator
accelerator = Accelerator(mixed_precision="fp16")

# Training parameters
model_name = "runwayml/stable-diffusion-v1-5"
instance_dir = "/kaggle/working/dreambooth/images"
output_dir = "/kaggle/working/dreambooth-model"

from diffusers import DreamBoothPipeline

from diffusers import StableDiffusionPipeline
from diffusers.training_utils import compute_snr

from diffusers import StableDiffusionPipeline
from diffusers import StableDiffusionTrainer

from diffusers import StableDiffusionPipeline, UNet2DConditionModel

# Load pipeline
pipe = DiffusionPipeline.from_pretrained(model_name, torch_dtype=torch.float16).to(accelerator.device)

# Train with DreamBooth
from diffusers import DreamBoothTrainer

trainer = DreamBoothTrainer(
    model=pipe,
    instance_data_dir=instance_dir,
    instance_prompt="a photo of sks person",  # change "sks person" to your custom token
    output_dir=output_dir,
    resolution=512,
    train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    lr_scheduler="constant",
    max_train_steps=800,   # adjust depending on how many images you have
)

trainer.train()


In [ ]:
!pip install --upgrade diffusers[training] transformers accelerate safetensors bitsandbytes

In [ ]:
!accelerate launch /usr/local/lib/python3.11/dist-packages/diffusers/examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="/kaggle/working/dreambooth/images" \
  --output_dir="/kaggle/working/dreambooth-model" \
  --instance_prompt="a photo of sks person" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-6 \
  --lr_scheduler="constant" \
  --max_train_steps=800 \
  --mixed_precision="fp16"

In [ ]:
!git clone https://github.com/huggingface/diffusers.git
%cd diffusers
!pip install -e .

In [ ]:
!accelerate launch examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="/kaggle/working/train_lora/images" \
  --output_dir="/kaggle/working/dreambooth-model" \
  --instance_prompt="a photo of sks person" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-6 \
  --lr_scheduler="constant" \
  --max_train_steps=800 \
  --mixed_precision="fp16"


In [ ]:
# Check GPU memory
!nvidia-smi

# Confirm images exist
!ls -lh /kaggle/working/dreambooth/images

# Test a single image
from PIL import Image
Image.open("/kaggle/working/dreambooth/images/<some_image>.png").show()

In [ ]:
!pip install --upgrade peft

In [ ]:
!pip install --upgrade diffusers transformers accelerate peft safetensors

In [ ]:
import peft
print(peft.__version__)

In [ ]:
pip show peft

In [ ]:
!accelerate launch \
  /usr/local/lib/python3.11/dist-packages/diffusers/examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
  --instance_data_dir=/kaggle/working/train_lora/images \
  --output_dir=/kaggle/working/dreambooth-model \
  --instance_prompt="a photo of sks person" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-6 \
  --lr_scheduler=constant \
  --max_train_steps=800 \
  --mixed_precision=fp16 \
  --report_to="tensorboard"

In [ ]:
mkdir -p ~/diffusers_examples/dreambooth
wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth.py -O ~/diffusers_examples/dreambooth/train_dreambooth.py

In [ ]:
!mkdir -p ~/diffusers_examples/dreambooth
!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth.py -O ~/diffusers_examples/dreambooth/train_dreambooth.py

In [ ]:
!accelerate launch ~/diffusers_examples/dreambooth/train_dreambooth.py \
    --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
    --instance_data_dir=/kaggle/working/train_lora/images \
    --output_dir=/kaggle/working/dreambooth-model \
    --instance_prompt="a photo of sks person" \
    --resolution=512 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate=5e-6 \
    --lr_scheduler=constant \
    --max_train_steps=800 \
    --mixed_precision=fp16


In [ ]:
import os

# Current folder path
old_path = "/kaggle/working/train_lora/images"

# New folder path
new_path = "/kaggle/working/train_lora/map"

# Rename the folder
os.rename(old_path, new_path)

print("Folder renamed successfully!")

In [ ]:
!accelerate launch ~/diffusers_examples/dreambooth/train_dreambooth.py \
    --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
    --instance_data_dir=/kaggle/input/globular-clusters \
    --output_dir=/kaggle/working/dreambooth-model \
    --instance_prompt="a photo of sks person" \
    --resolution=512 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate=5e-6 \
    --lr_scheduler=constant \
    --max_train_steps=800 \
    --mixed_precision=fp16


In [ ]:
import os

folder = "/kaggle/input/globular-clusters/"
for f in os.listdir(folder):
    if not f.lower().endswith((".png", ".jpg", ".jpeg")):
        print("Removing non-image file:", f)
        os.remove(os.path.join(folder, f))

In [ ]:
!accelerate launch ~/diffusers_examples/dreambooth/train_dreambooth.py \
    --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
    --instance_data_dir=/kaggle/input/globular-cluster \
    --output_dir=/kaggle/working/dreambooth-model \
    --instance_prompt="a photo of sks person" \
    --resolution=512 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate=5e-6 \
    --lr_scheduler=constant \
    --max_train_steps=800 \
    --mixed_precision=fp16


In [ ]:
!accelerate launch \
  --num_processes 2 \
  --mixed_precision="fp16" \
  /root/diffusers_examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/input/globular-cluster" \
  --instance_prompt="your_instance_prompt" \
  --class_prompt="your_class_prompt" \
  --output_dir="/kaggle/working/dreambooth_output" \
  --train_batch_size=2 \
  --gradient_accumulation_steps=1 \
  --max_train_steps=800 \
  --learning_rate=1e-4 \
  --resolution=512 \
  --lr_scheduler="constant" \
  --dataloader_num_workers=2 \
  --logging_dir="/kaggle/working/logs" \
  --save_model_every_n_steps=100 \
  --save_precision="fp16"


In [ ]:
!accelerate launch \
  --num_processes 2 \
  --mixed_precision="fp16" \
  /root/diffusers_examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="/kaggle/input/globular-cluster" \
  --instance_prompt="your_instance_prompt_here" \
  --output_dir="/kaggle/working/dreambooth_output" \
  --train_batch_size=2 \
  --gradient_accumulation_steps=1 \
  --max_train_steps=800 \
  --learning_rate=1e-4 \
  --resolution=512 \
  --dataloader_num_workers=2 \
  --logging_dir="/kaggle/working/logs" \
  --save_model_every_n_steps=100 \
  --save_precision="fp16"

In [ ]:
!accelerate launch \
  --num_processes 2 \
  --mixed_precision="fp16" \
  /root/diffusers_examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="/kaggle/input/globular-cluster" \
  --instance_prompt="your_instance_prompt_here" \
  --output_dir="/kaggle/working/dreambooth_output" \
  --train_batch_size=2 \
  --gradient_accumulation_steps=1 \
  --max_train_steps=800 \
  --learning_rate=1e-4 \
  --resolution=512 \
  --dataloader_num_workers=2 \
  --logging_dir="/kaggle/working/logs" \
  --checkpointing_steps=100 \
  --train_text_encoder \
  --enable_xformers_memory_efficient_attention


In [ ]:
!accelerate launch /root/diffusers_examples/dreambooth/train_dreambooth.py \
  --pretrained_model_name_or_path runwayml/stable-diffusion-v1-5 \
  --instance_data_dir /kaggle/input/your-instance-images \
  --class_data_dir /kaggle/input/your-class-images \
  --instance_prompt "a photo of my subject" \
  --class_prompt "a photo of a person" \
  --output_dir /kaggle/working/dreambooth_output \
  --train_batch_size 1 \
  --mixed_precision fp16 \
  --num_train_epochs 3 \
  --gradient_accumulation_steps 1 \
  --checkpointing_steps 50 \
  --resolution 512 \
  --train_text_encoder

In [ ]:
!pip install --upgrade diffusers transformers accelerate xformers safetensors

from pathlib import Path

# Paths
instance_dir = Path("/kaggle/input/globular-cluster")
output_dir = Path("/kaggle/working/dreambooth_output")

# Model name
pretrained_model = "runwayml/stable-diffusion-v1-5"

# Training parameters
train_args = {
    "pretrained_model_name_or_path": pretrained_model,
    "instance_data_dir": str(instance_dir),
    "instance_prompt": "a realistic globular star cluster, night sky, high resolution",
    "output_dir": str(output_dir),
    "resolution": 512,
    "train_batch_size": 2,      # increase if GPU memory allows
    "gradient_accumulation_steps": 4,
    "learning_rate": 5e-6,
    "max_train_steps": 800,     # adjust for dataset size
    "mixed_precision": "fp16",
    "save_interval": 200,       # save every N steps
    "enable_xformers_memory_efficient_attention": True
}

# Launch DreamBooth training
!accelerate launch /kaggle/working/diffusers_examples/dreambooth/train_dreambooth.py \
    --pretrained_model_name_or_path {train_args['pretrained_model_name_or_path']} \
    --instance_data_dir {train_args['instance_data_dir']} \
    --instance_prompt "{train_args['instance_prompt']}" \
    --output_dir {train_args['output_dir']} \
    --resolution {train_args['resolution']} \
    --train_batch_size {train_args['train_batch_size']} \
    --gradient_accumulation_steps {train_args['gradient_accumulation_steps']} \
    --learning_rate {train_args['learning_rate']} \
    --max_train_steps {train_args['max_train_steps']} \
    --mixed_precision {train_args['mixed_precision']} \
    --enable_xformers_memory_efficient_attention

In [ ]:
pip install xformers==0.0.31 -f https://download.pytorch.org/whl/cu124/torch_stable.html

In [ ]:
from pathlib import Path
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, AutoencoderKL, DPMSolverMultistepScheduler
from diffusers import StableDiffusionDreamBoothPipeline, DreamBoothTrainingArguments, DreamBoothTrainer
from transformers import CLIPTextModel, CLIPTokenizer
import torch

# Paths
instance_dir = Path("/kaggle/input/globular-cluster")  # your globular cluster images
class_dir = Path("/kaggle/working/dataset/class_images")  # optional, can leave empty
output_dir = Path("/kaggle/working/dreambooth_model")

# Model checkpoint
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

# Tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path)
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path)

# UNet and Autoencoder
vae = AutoencoderKL.from_pretrained(pretrained_model_name_or_path)
unet = UNet2DConditionModel.from_pretrained(pretrained_model_name_or_path)

# Training args
training_args = DreamBoothTrainingArguments(
    instance_data_dir=str(instance_dir),
    class_data_dir=str(class_dir) if class_dir.exists() else None,
    output_dir=str(output_dir),
    learning_rate=1e-4,
    max_train_steps=800,       # adjust according to GPU memory
    gradient_accumulation_steps=1,
    train_batch_size=1,
    mixed_precision="fp16",
    logging_dir=str(output_dir / "logs"),
    save_interval=200,
    save_preview=True,
    resolution=512,
)

# Initialize trainer
trainer = DreamBoothTrainer(
    model=StableDiffusionPipeline.from_pretrained(pretrained_model_name_or_path).to("cuda"),
    args=training_args,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    unet=unet,
    vae=vae,
)

# Start training
trainer.train()

# Once training is done, load trained model for generation
pipeline = StableDiffusionPipeline.from_pretrained(str(output_dir), torch_dtype=torch.float16).to("cuda")
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)

# Generate a new random globular cluster
prompt = "A realistic image of a new globular cluster in deep space, star cluster, bright, colorful"
image = pipeline(prompt, guidance_scale=7.5, num_inference_steps=50).images[0]

# Save output
image.save("/kaggle/working/new_globular_cluster.png")


In [ ]:
RuntimeError: Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Could not import module 'CLIPImageProcessor'. Are this object's requirements defined correctly?

In [ ]:
pip install diffusers==0.15.1 transformers==4.31.0

In [ ]:
from pathlib import Path
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, AutoencoderKL, DPMSolverMultistepScheduler
from diffusers import StableDiffusionDreamBoothPipeline, DreamBoothTrainingArguments, DreamBoothTrainer
from transformers import CLIPTextModel, CLIPTokenizer
import torch

# Paths
instance_dir = Path("/kaggle/input/globular-cluster")  # your globular cluster images
class_dir = Path("/kaggle/working/dataset/class_images")  # optional, can leave empty
output_dir = Path("/kaggle/working/dreambooth_model")

# Model checkpoint
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

# Tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path)
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path)

# UNet and Autoencoder
vae = AutoencoderKL.from_pretrained(pretrained_model_name_or_path)
unet = UNet2DConditionModel.from_pretrained(pretrained_model_name_or_path)

# Training args
training_args = DreamBoothTrainingArguments(
    instance_data_dir=str(instance_dir),
    class_data_dir=str(class_dir) if class_dir.exists() else None,
    output_dir=str(output_dir),
    learning_rate=1e-4,
    max_train_steps=800,       # adjust according to GPU memory
    gradient_accumulation_steps=1,
    train_batch_size=1,
    mixed_precision="fp16",
    logging_dir=str(output_dir / "logs"),
    save_interval=200,
    save_preview=True,
    resolution=512,
)

# Initialize trainer
trainer = DreamBoothTrainer(
    model=StableDiffusionPipeline.from_pretrained(pretrained_model_name_or_path).to("cuda"),
    args=training_args,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    unet=unet,
    vae=vae,
)

# Start training
trainer.train()

# Once training is done, load trained model for generation
pipeline = StableDiffusionPipeline.from_pretrained(str(output_dir), torch_dtype=torch.float16).to("cuda")
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)

# Generate a new random globular cluster
prompt = "A realistic image of a new globular cluster in deep space, star cluster, bright, colorful"
image = pipeline(prompt, guidance_scale=7.5, num_inference_steps=50).images[0]

# Save output
image.save("/kaggle/working/new_globular_cluster.png")

In [ ]:
pip uninstall diffusers transformers huggingface_hub -y

In [ ]:
pip install diffusers==0.15.1 transformers==4.31.0 huggingface_hub==0.18.1

In [ ]:
from pathlib import Path
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, AutoencoderKL, DPMSolverMultistepScheduler
from diffusers import StableDiffusionDreamBoothPipeline, DreamBoothTrainingArguments, DreamBoothTrainer
from transformers import CLIPTextModel, CLIPTokenizer
import torch

# Paths
instance_dir = Path("/kaggle/input/globular-cluster")  # your globular cluster images
class_dir = Path("/kaggle/working/dataset/class_images")  # optional, can leave empty
output_dir = Path("/kaggle/working/dreambooth_model")

# Model checkpoint
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

# Tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name_or_path)
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name_or_path)

# UNet and Autoencoder
vae = AutoencoderKL.from_pretrained(pretrained_model_name_or_path)
unet = UNet2DConditionModel.from_pretrained(pretrained_model_name_or_path)

# Training args
training_args = DreamBoothTrainingArguments(
    instance_data_dir=str(instance_dir),
    class_data_dir=str(class_dir) if class_dir.exists() else None,
    output_dir=str(output_dir),
    learning_rate=1e-4,
    max_train_steps=800,       # adjust according to GPU memory
    gradient_accumulation_steps=1,
    train_batch_size=1,
    mixed_precision="fp16",
    logging_dir=str(output_dir / "logs"),
    save_interval=200,
    save_preview=True,
    resolution=512,
)

# Initialize trainer
trainer = DreamBoothTrainer(
    model=StableDiffusionPipeline.from_pretrained(pretrained_model_name_or_path).to("cuda"),
    args=training_args,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    unet=unet,
    vae=vae,
)

# Start training
trainer.train()

# Once training is done, load trained model for generation
pipeline = StableDiffusionPipeline.from_pretrained(str(output_dir), torch_dtype=torch.float16).to("cuda")
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)

# Generate a new random globular cluster
prompt = "A realistic image of a new globular cluster in deep space, star cluster, bright, colorful"
image = pipeline(prompt, guidance_scale=7.5, num_inference_steps=50).images[0]

# Save output
image.save("/kaggle/working/new_globular_cluster.png")

In [ ]:
!pip uninstall -y diffusers transformers

In [ ]:
!pip install diffusers==0.19.1 transformers==4.33.2 huggingface_hub==0.18.1

In [ ]:
!pip uninstall -y diffusers transformers huggingface_hub
!pip install diffusers==0.19.1 transformers==4.33.2 huggingface_hub --upgrade

In [1]:
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, AutoencoderKL
from diffusers import StableDiffusionDreamBoothPipeline, DreamBoothTrainingArguments, DreamBoothTrainer
from transformers import CLIPTextModel, CLIPTokenizer
import torch

ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/usr/local/lib/python3.11/dist-packages/huggingface_hub/__init__.py)